In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    pairwise_distances_argmin,
    classification_report,
    accuracy_score
)

In [2]:
# Load datasets
train = pd.read_csv("../training_image_features_grouped.csv")
test = pd.read_csv("../testing_image_features_grouped.csv")

TARGET = "Condition_Class"
CLASSES = ["Excellent", "Good", "Fair", "Poor"]

# Use numeric image features; exclude target-related scores
exclude = {
    "Condition_Score",
    "Damage_Score",
    "Alternative_Use_Suitability"
}

In [3]:
from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X_train = train[features].replace([np.inf, -np.inf], np.nan)
X_test = test[features].replace([np.inf, -np.inf], np.nan)

# Preprocessing
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train = scaler.fit_transform(imputer.fit_transform(X_train))
X_test = scaler.transform(imputer.transform(X_test))

# Fit hierarchical clustering on a training subset
np.random.seed(42)
sample_size = min(2500, len(X_train))
indices = np.random.choice(
    len(X_train), sample_size, replace=False
)

model = AgglomerativeClustering(
    n_clusters=4,
    linkage="ward"
)

cluster_labels = model.fit_predict(X_train[indices])


In [4]:
# Calculate cluster centroids safely
centroids = []

for cluster in range(4):
    cluster_points = X_train[indices][
        cluster_labels == cluster
    ]

    if len(cluster_points) > 0:
        centroids.append(cluster_points.mean(axis=0))
    else:
        centroids.append(X_train[indices].mean(axis=0))

centroids = np.vstack(centroids)

# Assign training and testing samples to nearest centroid
train_clusters = pairwise_distances_argmin(
    X_train, centroids
)

test_clusters = pairwise_distances_argmin(
    X_test, centroids
)

In [5]:
# Map clusters to majority condition class
cluster_mapping = {}

for cluster in range(4):
    labels = train.loc[
        train_clusters == cluster, TARGET
    ]

    if len(labels) > 0:
        cluster_mapping[cluster] = labels.mode().iloc[0]
    else:
        cluster_mapping[cluster] = train[TARGET].mode().iloc[0]

# Predict test classes
y_pred = np.array([
    cluster_mapping[c] for c in test_clusters
])

y_test = test[TARGET].astype(str)

# Classification metrics
print("Hierarchical Clustering")
print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    labels=CLASSES,
    zero_division=0
))

Hierarchical Clustering
Accuracy: 0.6762481089258698
              precision    recall  f1-score   support

   Excellent       0.00      0.00      0.00        32
        Good       0.00      0.00      0.00       586
        Fair       0.68      1.00      0.81      1788
        Poor       0.00      0.00      0.00       238

    accuracy                           0.68      2644
   macro avg       0.17      0.25      0.20      2644
weighted avg       0.46      0.68      0.55      2644



In [6]:
# Clustering is unsupervised: the accuracy above comes from mapping each cluster to its
# majority class, so compare it with the majority-class baseline and use label-free metrics.
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score, f1_score)
_y = test[TARGET].astype(str)
_X = X_train_pca if "X_train_pca" in globals() else X_train
print("Majority-class baseline accuracy:", round(_y.value_counts(normalize=True).max(), 4))
print("Macro F1 of cluster-to-class mapping:", round(f1_score(_y, y_pred, average="macro", zero_division=0), 4))
print("Adjusted Rand Index vs true labels (test):", round(adjusted_rand_score(_y, test_clusters), 4))
print("Normalized Mutual Information (test):", round(normalized_mutual_info_score(_y, test_clusters), 4))
print("Silhouette (train sample):", round(silhouette_score(_X, train_clusters, sample_size=2000, random_state=42), 4))


Majority-class baseline accuracy: 0.6762
Macro F1 of cluster-to-class mapping: 0.2017
Adjusted Rand Index vs true labels (test): 0.0505
Normalized Mutual Information (test): 0.0783
Silhouette (train sample): 0.1397


In [7]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Fit PCA on training data
pca = PCA(n_components=2)

pca.fit(X_train)

# Transform testing data
X_test_pca = pca.transform(X_test)

plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    X_test_pca[:, 0],
    X_test_pca[:, 1],
    c=test_clusters,
    cmap="viridis",
    s=15,
    alpha=0.8
)

plt.title("Hierarchical Clustering Visualization")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.colorbar(scatter, label="Cluster")

plt.tight_layout()
plt.show()

print(
    "Explained Variance:",
    pca.explained_variance_ratio_.sum()
)

Explained Variance: 0.46925645851253023


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_14620\1060756138.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as sch
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Use the feature matrix from your hierarchical clustering implementation
# X_train should contain the numerical training features

# Sample rows for a readable dendrogram
sample_size = min(150, len(X_train))
X_sample = X_train[:sample_size]

# Ward linkage
linkage_matrix = sch.linkage(X_sample, method="ward")

# Plot dendrogram
plt.figure(figsize=(14, 6))
sch.dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=30,
    leaf_rotation=90,
    leaf_font_size=8,
    show_contracted=True
)

plt.title("Hierarchical Clustering Dendrogram (Ward Linkage)")
plt.xlabel("Cluster / Sample")
plt.ylabel("Euclidean Distance")
plt.tight_layout()
plt.show()

C:\Users\Ryan John\AppData\Local\Temp\ipykernel_14620\3424555164.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
